# Runnable 실습진행.
### 실습 1. 함수를 Runnable로 바꾸기 (API 키 불필요)


In [2]:
from langchain_core.runnables import RunnableLambda

def add_one(x):
    return x+1

r_add = RunnableLambda(add_one)
print(r_add.invoke(3))
print(r_add.batch([1,2,3]))

4
[2, 3, 4]


In [3]:
from langchain_core.runnables import RunnableLambda

r_add = RunnableLambda(lambda x:x+1)
r_double = RunnableLambda(lambda x:x*2)



### 실습 2. `|`로 연결하기 (API 키 불필요)

In [4]:
chain_a = r_add|r_double
chain_b = r_double | r_add

print(chain_a.invoke(3))
print(chain_b.invoke(3))
print(type(chain_a)) # <class 'langchain_core.runnables.base.RunnableSequence'>
print(type(chain_b)) # <class 'langchain_core.runnables.base.RunnableSequence'>

8
7
<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.runnables.base.RunnableSequence'>


### 실습 3. RunnableParallel과 RunnablePassthrough (API 키 불필요)

In [48]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

analyzer = RunnableParallel(
        original = RunnablePassthrough(),
        upper = RunnableLambda(lambda x: x.upper()),
        length = RunnableLambda(lambda x:len(x)),
)

print(analyzer.invoke("hello"))


{'original': 'hello', 'upper': 'HELLO', 'length': 5}


### 실습 4.  LLM 체인 만들기 (API 키 필요)  

| 단계 | 들어오는 것 | 나가는 것 |
|---|---|---|
| prompt | {"topic": "인공지능"} (딕셔너리) | 완성된 프롬프트 메시지 |
| model | 프롬프트 메시지 | AIMessage (답변 + 부가정보) |
| parser | AIMessage| 문자열 |

부품을 하나씩 실행해 단게별 출력을 직접 해보세요.

In [9]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os
from dotenv import load_dotenv
load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")

# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")





API 키 로드: sk-proj...
✅ OpenAI 클라이언트 준비 완료


In [46]:
# LangChain 방식 — 부품 3개를 만들어 파이프로 연결
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [22]:
# 부품 1: 모델
llm = ChatOpenAI(model="gpt-4o-mini")
# 부품 3: 파서
parser = StrOutputParser()
# 부품 2: 프롬프트 (변수 자리 {role}, {question})
prompt = ChatPromptTemplate.from_template(
    "당신은 대학교수입니다. 질문:{question}")

# 부품 3: 체인 = 프롬프트 | 모델 | 파서
chain = prompt | llm | parser

chain.invoke({"question":"반복학습"})

"반복학습(repetitive learning)은 학습자가 정보를 기억하고 이해하기 위해 특정 내용을 여러 번 반복하는 학습 방법입니다. 이 방법은 다음과 같은 이유로 효과적일 수 있습니다.\n\n1. **기억 강화**: 정보를 반복적으로 노출받으면 장기 기억에 저장될 확률이 높아집니다. 이는 '지속적 재인'의 원칙에 따라 이루어집니다.\n\n2. **이해도 향상**: 반복 학습을 통해 주제를 여러 번 접하면서 깊이 있는 이해를 도울 수 있습니다. 처음에는 표면적인 이해에서 시작하여 이후 반복을 통해 더 깊은 통찰을 얻을 수 있습니다.\n\n3. **자신감 증진**: 반복학습을 통해 학습자는 자신이 알고 있는 내용을 확인하고, 시험이나 평가에서 자신감을 가질 수 있습니다.\n\n4. **학습 패턴 설계**: 반복학습은 특정 시간 간격으로 복습하는 구조적 접근(예: spaced repetition)을 통해 이루어질 수 있습니다. 이는 학습자에게 효율적으로 정보를 정리하고 소화할 기회를 제공합니다.\n\n반복학습의 효과를 극대화하기 위해서는 단순히 같은 내용을 반복하는 것 외에도, 예제나 문제풀이, 다른 관점에서의 접근 등 다양한 방법을 조합해 나가는 것이 좋습니다. 학습자의 흥미를 유지하기 위해 다양한 매체나 자료를 사용하는 것도 효과적입니다. \n\n궁금한 점이나 더 구체적인 질문이 있으시면 말씀해 주세요!"

In [25]:
results = chain.batch([{"question":"블럭체인"},{"question":"클라우드"}])

for chunk in chain.stream({"question":"양자컴퓨터"}):
    print(chunk, end="", flush=True)

양자컴퓨터는 양자역학의 원리를 이용하여 정보를 처리하는 컴퓨터입니다. 전통적인 컴퓨터는 비트(bit)를 사용하여 정보를 표현하는 반면, 양자컴퓨터는 큐비트(qubit)를 사용합니다. 큐비트는 0과 1의 상태를 동시에 가질 수 있는 중첩(superposition) 상태를 통해 대량의 계산을 병렬로 수행할 수 있는 특징이 있습니다.

양자컴퓨터의 주요 특징 중 몇 가지는 다음과 같습니다:

1. **중첩(Superposition)**: 큐비트는 여러 상태를 동시에 가질 수 있어 복잡한 문제를 해결할 때 더 많은 경로를 탐색할 수 있습니다.

2. **얽힘(Entanglement)**: 두 개 이상의 큐비트가 서로 얽힐 수 있어, 하나의 큐비트의 상태를 측정하면 다른 큐비트의 상태도 즉시 결정되는 특성을 가집니다. 이를 통해 정보 전송의 효율성을 높일 수 있습니다.

3. **양자 간섭(Quantum Interference)**: 여러 경로로 계산된 결과들을 조합하여 원하는 결과를 강조하고 원하지 않는 결과를 제거할 수 있습니다.

양자컴퓨터는 특정 문제들, 예를 들어 소인수 분해, 최적화 문제, 양자 시뮬레이션 등에서 기존의 클래식 컴퓨터보다 훨씬 더 빠른 성능을 보여줄 수 있습니다. 하지만 현재 양자컴퓨터 기술은 아직 상용화 단계에는 이르지 않았으며, 많은 기술적 도전과제가 남아 있습니다.

양자컴퓨터에 대한 연구는 아직 초기 단계이지만, 이 기술이 발전함에 따라 인공지능, 물리학, 화학, 금융 등 여러 분야에 혁신적인 변화를 가져올 가능성이 큽니다. 추가적인 질문이나 특정한 주제에 대한 설명이 필요하시면 말씀해 주세요!

# 도전과제

**과제 1 (기초, API 키 불필요)** — 문자열을 받아 양쪽 공백을 없애고, 대문자로 바꾼 뒤, 끝에 `!`를 붙이는 체인을 만드세요.

```python
# 입력: "  hello world  "
# 출력: "HELLO WORLD!"
```

In [29]:
s_del = RunnableLambda(lambda x: x.strip())
s_up = RunnableLambda(lambda x: x.upper())
s_add = RunnableLambda(lambda x: x+'!')

chain_str = s_del|s_up|s_add

print(chain_str.invoke("  hello world  "))

HELLO WORLD!


**과제 2 (중급, API 키 불필요)** — 숫자 리스트를 받아 합계, 평균, 최댓값을 한 번에 딕셔너리로 돌려주는 Runnable을 만드세요.

```python
# 입력: [10, 20, 30]
# 출력: {'total': 60, 'average': 20.0, 'max': 30}
```

In [1]:
from langchain_core.runnables import RunnableParallel, RunnableLambda
calcu =RunnableParallel(
    total = RunnableLambda(lambda x: sum(x)),
    average = RunnableLambda(lambda x: sum(x)/len(x)),
    max = RunnableLambda(lambda x: max(x)),
    )


print(calcu.invoke([10, 20, 30]))

#print({'total':total.invoke([10, 20, 30]),'average':average.invoke([10, 20, 30]),'max':max.invoke([10, 20, 30])})

{'total': 60, 'average': 20.0, 'max': 30}


**과제 3 (심화, API 키 필요)** — 주제 하나를 넣으면 ① 한 줄 설명과 ② 관련 키워드 3개를 동시에 만들어 딕셔너리로 돌려주는 체인을 만드세요.

```python
# 입력: {"topic": "머신러닝"}
# 출력: {'summary': '...', 'keywords': '...'}
```

힌트: 프롬프트를 두 개 만들고 `RunnableParallel`로 묶으세요.

In [ ]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os
from dotenv import load_dotenv
load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")

# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")

In [13]:
# LangChain 방식 — 부품 3개를 만들어 파이프로 연결
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

# 부품 1: 모델
llm = ChatOpenAI(model="gpt-4o-mini")
# 부품 3: 파서
parser = StrOutputParser()
# 부품 2: 프롬프트 (변수 자리 {role}, {question})
prompt = ChatPromptTemplate.from_template(
    "당신은 대학교수입니다. 질문:{question}")

# 부품 3: 체인 = 프롬프트 | 모델 | 파서
chain = prompt | llm | parser


# 프롬프트 요약
summary = ChatPromptTemplate.from_template(
    "질문:{question}에 대한 답변을 한줄로 요약해.")

# 프롬프트 키워드
keywords =ChatPromptTemplate.from_template(
    "질문:{question}에 대한 핵심 키워드 3개를 뽑아 ','로 구분해 출력해 ")

# 체이닝 진행.
chain_s = summary | llm | parser

chain_k = keywords | llm | parser
sumprompt = RunnableParallel(summ = chain_s,
                             keyword = chain_k)

print(sumprompt.invoke({"question":"반복학습"}))


{'summ': '반복학습은 정보를 지속적으로 복습함으로써 기억을 강화하고 학습 효과를 높이는 방법이다.', 'keyword': '반복, 기억, 연습'}
